# ARC-AGI-3: Systematic Exploration Baseline

This first entry uses the official ARC runtime and competition gateway. It performs a deterministic, reproducible sweep of each game's available button actions and an 8×8 grid of click points. It has no external API calls or model dependencies, so it runs with internet disabled.

This is a baseline for establishing a valid end-to-end submission and collecting scorecard feedback. It does not yet infer game rules from observations.

License: MIT No Attribution (MIT-0). The original code in this notebook is dedicated to the public domain under the MIT-0 terms.

In [ ]:
from pathlib import Path
import subprocess, sys

COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
WHEELHOUSE = COMP_ROOT / 'arc_agi_3_wheels'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
    '--no-warn-conflicts', '--disable-pip-version-check',
    '--find-links', str(WHEELHOUSE), 'arc-agi'
], check=True)
print('Installed the competition-bundled ARC runtime.')

In [ ]:
import os, time
from urllib.request import urlopen
from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState

IS_COMPETITION_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in {'1', 'true'}
if IS_COMPETITION_RUN:
    BASE_URL = 'http://gateway:8001/'
    os.environ['ARC_API_KEY'] = 'test-key-123'
    os.environ['ONLY_RESET_LEVELS'] = 'true'
    deadline = time.monotonic() + 600
    while True:
        try:
            with urlopen(BASE_URL + 'api/games', timeout=10) as response:
                if response.status < 500:
                    break
        except Exception:
            if time.monotonic() >= deadline:
                raise RuntimeError('Kaggle competition gateway did not start in time.')
            time.sleep(5)
    arcade = Arcade(arc_api_key='test-key-123', arc_base_url=BASE_URL,
                    operation_mode=OperationMode.COMPETITION, environments_dir='',
                    recordings_dir='/kaggle/working/recordings')
else:
    env_dir = COMP_ROOT / 'environment_files'
    arcade = Arcade(operation_mode=OperationMode.OFFLINE,
                    environments_dir=str(env_dir),
                    recordings_dir='/kaggle/working/recordings')
games = sorted({info.game_id for info in arcade.available_environments})
if not games:
    raise RuntimeError('ARC runtime found no games to play.')
print(f'Run mode: {"competition" if IS_COMPETITION_RUN else "offline preview"}; games: {len(games)}')

In [ ]:
# A compact sweep: try each simple control several times, then scan click points.
SIMPLE_ACTIONS = [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
                  GameAction.ACTION4, GameAction.ACTION5, GameAction.ACTION7]
GRID = list(range(4, 64, 8))
CLICK_POINTS = [(x, y) for y in GRID for x in GRID]
PLAN = [(a, None, None) for a in SIMPLE_ACTIONS for _ in range(4)]
PLAN += [(GameAction.ACTION6, x, y) for x, y in CLICK_POINTS]
MAX_ACTIONS_PER_GAME = 256

def play_game(game_id, scorecard_id):
    env = arcade.make(game_id, scorecard_id=scorecard_id, include_frame_data=True)
    if env is None:
        print(f'{game_id}: environment unavailable')
        return 0, 0
    frame = env.step(GameAction.RESET, data={'game_id': game_id})
    steps = 0
    while frame is not None and steps < MAX_ACTIONS_PER_GAME:
        if frame.state == GameState.WIN:
            break
        if frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            action, x, y = GameAction.RESET, None, None
        else:
            allowed = set(frame.available_actions or range(1, 8))
            action, x, y = PLAN[steps % len(PLAN)]
            # Skip actions the environment currently disallows.
            for offset in range(len(PLAN)):
                candidate = PLAN[(steps + offset) % len(PLAN)]
                if candidate[0].value in allowed:
                    action, x, y = candidate
                    break
        payload = {'game_id': game_id}
        if action.is_complex():
            payload.update(x=x, y=y)
        action.set_data(payload)
        frame = env.step(action, data=action.action_data.model_dump())
        steps += 1
    levels = int(getattr(frame, 'levels_completed', 0)) if frame is not None else 0
    print(f'{game_id}: actions={steps}, levels_completed={levels}')
    return steps, levels

scorecard_id = arcade.open_scorecard(tags=['agent', 'systematic-exploration-baseline'])
results = []
try:
    for index, game_id in enumerate(games, 1):
        try:
            results.append((game_id, *play_game(game_id, scorecard_id)))
        except Exception as exc:
            print(f'{game_id}: error: {type(exc).__name__}: {exc}')
            results.append((game_id, 0, 0))
        if index % 10 == 0:
            print(f'Progress: {index}/{len(games)} games')
finally:
    scorecard = arcade.close_scorecard(scorecard_id)
print('Finished. Scorecard:')
if scorecard is not None:
    print(scorecard.model_dump_json(indent=2))
print(f'Games attempted: {len(results)}; levels completed: {sum(r[2] for r in results)}')

In [ ]:
# Kaggle requires a named output artifact to enable code submission. The ARC gateway
# computes the real evaluation from the agent's run; this file only satisfies the
# normal preview-run output contract.
if not IS_COMPETITION_RUN:
    import pandas as pd
    placeholder = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    placeholder.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Wrote the required preview submission.parquet artifact.')
